### Export coefficeints to file

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from chimorse.config import (
    FigureContext, get_colors, PLOT_PARAMS,
    load_molecule_info
)
from chimorse.datasets import ensure_reference_data
from chimorse.dataio import load_data, export_chimorse_single_json, export_chimorse_combined_json
from chimorse.fitting import _parse_prune_arg, extract_reduced_coeffs, fit_alpha_values
from chimorse.fourier import create_matrix_lsqt_2d
from chimorse.analysis import extract_energy_minimums, get_symm_chi

plt.rcParams.update(PLOT_PARAMS)

In [ ]:
molecule_name = 'PA'
interaction = 'OA'
zero_zeta = True

alpha_fit = True

harmonic_ceils = {'EP': (8, 3), 'EA': (8, 3), 'OP': (20, 1), 'OA': (20, 1)}

data_root = Path("../data")
data_dir = ensure_reference_data(molecule_name, data_root=data_root)
molecule = load_molecule_info(
    molecule_name,
    metadata_path=data_dir / "metadata.json",
)

df_org = load_data(molecule, interaction, zero_zeta)
colors = get_colors()

ctx_org = FigureContext(base="Figures", molecule=molecule.name, data_type="org", interaction=interaction)
ctx_model = FigureContext(base="Figures", molecule=molecule.name, data_type="model", interaction=interaction)
ctx_compare = FigureContext(base="Figures", molecule=molecule.name, data_type="compare", interaction=interaction)

In [ ]:
from pathlib import Path
import numpy as np

# ============================================================
# 1. Select the model
# ============================================================

model_list = [
    "full_pruned",
    "full_unpruned",
    "compact_pruned",
    "compact_unpruned",
]

model_name = model_list[2]  # Change this index to select the model

alpha_fit = model_name.startswith("full_")
is_pruned = model_name.endswith("_pruned")
is_compact = model_name.startswith("compact_")

model_dir = Path("models") / model_name
model_dir.mkdir(parents=True, exist_ok=True)

print(f"Model: {model_name}")
print(f"Fourier alpha: {alpha_fit}")
print(f"Pruning: {is_pruned}")

harmonic_ceils_full = {
    'EP': (8, 3), 'EA': (8, 3), 'OP': (20, 1), 'OA': (20, 1)
}

harmonic_ceils_compact = {
    'EP': (8, 1), 'EA': (8, 1), 'OP': (20, 1), 'OA': (20, 1)
}

harmonic_ceils = (
    harmonic_ceils_compact
    if is_compact
    else harmonic_ceils_full
)
# ============================================================
# 2. Pruning settings  # Change these numbers based on RMSE convergence
# ============================================================
compact_top_n = {
    "EP": {"D": 22, "re": 24},
    "EA": {"D": 30, "re": 35},
    "OP": {"D": 25, "re": 35},
    "OA": {"D": 42, "re": 70},
}

full_top_n = {
    "EP": {"D": 25, "re": 40, "alpha": 50},
    "EA": {"D": 30, "re": 40, "alpha": 50},
    "OP": {"D": 20, "re": 30, "alpha": 30},
    "OA": {"D": 50, "re": 55, "alpha": 55},
}

top_n = compact_top_n if is_compact else full_top_n

# ============================================================
# 3. Fit and optionally prune each interaction class
# ============================================================
for interaction in ["EP", "EA", "OP", "OA"]:

    print(f"\nProcessing {interaction} ...")

    df_org = load_data(molecule, interaction, zero_zeta)

    E_min_df = extract_energy_minimums(df_org, r_max=12)

    D = -E_min_df["e"]
    re = E_min_df["r"]

    chi_rad = np.deg2rad(E_min_df["chi"])
    psi_rad = np.deg2rad(E_min_df["psi"])

    h_chi, h_psi = harmonic_ceils[interaction]
    symm_chi = get_symm_chi(interaction)

    A, labels = create_matrix_lsqt_2d(
        h_chi,
        h_psi,
        chi_rad,
        psi_rad,
        symm_chi,
        molecule.screw_step,
    )

    # --------------------------------------------------------
    # Fit all required parameters (independent of pruning)
    # --------------------------------------------------------
    D_coeff, *_ = np.linalg.lstsq(A, D, rcond=None)
    re_coeff, *_ = np.linalg.lstsq(A, re, rcond=None)

    alpha_coeff = None
    alpha_vals = None

    if alpha_fit:
        alpha_vals = fit_alpha_values(df_org, interaction)
        alpha_coeff, *_ = np.linalg.lstsq(A, alpha_vals, rcond=None)

    # --------------------------------------------------------
    # Apply pruning only for pruned models
    # --------------------------------------------------------
    if is_pruned:

        D_coeff, _ = extract_reduced_coeffs(A, D, top_n=top_n[interaction]["D"])
        re_coeff, _ = extract_reduced_coeffs(A, re, top_n=top_n[interaction]["re"])

        if alpha_fit:
            alpha_coeff, _ = extract_reduced_coeffs(A, alpha_vals, top_n=top_n[interaction]["alpha"])

    # --------------------------------------------------------
    # Print coefficient counts
    # --------------------------------------------------------
    print(f"D: {np.count_nonzero(D_coeff)} / {len(D_coeff)}")
    print(f"re: {np.count_nonzero(re_coeff)} / {len(re_coeff)}")

    if alpha_fit:
        print(
            f"alpha: {np.count_nonzero(alpha_coeff)} "
            f"/ {len(alpha_coeff)}"
        )
    else:
        print("alpha: constant (1.1)")

    # --------------------------------------------------------
    # Export individual interaction model
    # --------------------------------------------------------
    export_chimorse_single_json(
        filepath=str(model_dir / f"{interaction}.json"),
        h_chi=h_chi,
        h_psi=h_psi,
        symm_chi=symm_chi,
        screw_step=molecule.screw_step,
        D_coeff=D_coeff,
        re_coeff=re_coeff,
        fixed_alpha=1.1 if not alpha_fit else None,
        alpha_coeff=alpha_coeff,
        cutoff=12.0,
    )

# ============================================================
# 4. Combine all four interaction classes
# ============================================================
export_chimorse_combined_json(
    target_file=str(model_dir / "chimorse_all.json"),
    source_directory=str(model_dir),
)

print(f"\nExport completed: {model_dir / 'chimorse_all.json'}")


In [ ]:
import json
from pathlib import Path

base = Path("/home/hadis/chimorse/examples/models")

for model_name in ["compact", "full"]:
    for pruned_name in ["pruned", "unpruned"]:
        path = base / f"{model_name}_{pruned_name}" / "chimorse_all.json"
        data = json.loads(path.read_text())

        print(f"\n=== {model_name.upper()} {pruned_name.upper()} ===")

        for interaction in ["EP", "EA", "OP", "OA"]:
            model = data["interactions"][interaction]
            params = model["parameters"]
            basis_size = len(model["basis_terms"])

            active = [False] * basis_size
            counts = {}

            for name in ["D", "re", "alpha"]:
                p = params[name]

                if p["type"] == "constant":
                    counts[name] = "constant"
                    continue

                coeffs = p["coefficients"]
                counts[name] = sum(c != 0 for c in coeffs)

                for i, c in enumerate(coeffs):
                    if c != 0:
                        active[i] = True

            print(
                f"{interaction}: "
                f"basis={basis_size}, "
                f"active union={sum(active)}, "
                f"D={counts['D']}, "
                f"re={counts['re']}, "
                f"alpha={counts['alpha']}"
            )

---

### Calculate Force and export Potential/ Force to C++

In [ ]:
import sympy as sp
from chimorse.models import Morse_1D

r, dphi = sp.symbols('r deltaPhi', real=True)
D, re, a, h, alpha = sp.symbols('D re a h alpha', real=True)

morse_part = D * (sp.exp(-2 * a * (r - re)) - 2 * sp.exp(-a * (r - re)))
fourier_part = sp.cos(h * dphi + alpha) # example

U_term = morse_part * fourier_part

F_radial = -sp.diff(U_term, r)
F_angular = -sp.diff(U_term, dphi)

print("====== C++ Code for Potential Energy ======")
print(sp.ccode(U_term, assign_to="energy_term"))

print("\n====== C++ Code for Radial Force ======")
print(sp.ccode(F_radial, assign_to="radial_term"))

print("\n====== C++ Code for Angular Force ======")
print(sp.ccode(F_angular, assign_to="angular_term"))